# Text Analysis with spaCy
Upload your `.txt` files and get Word Count, Sentence Count, and Average Sentence Length per file.

In [ ]:
# Cell 1: Install dependencies
!pip install spacy openpyxl -q
!python -m spacy download en_core_web_sm -q

In [ ]:
# Cell 2: Upload your .txt files
from google.colab import files

uploaded = files.upload()  # Select one or multiple .txt files
print(f"Uploaded files: {list(uploaded.keys())}")

In [ ]:
# Cell 3: Analyze and export to Excel
import spacy
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

nlp = spacy.load("en_core_web_sm")

rows = []

for filename, content in uploaded.items():
    text = content.decode("utf-8")
    doc = nlp(text)

    sentences = [sent for sent in doc.sents]
    words = [token for token in doc if not token.is_punct and not token.is_space]

    sentence_count = len(sentences)
    word_count = len(words)
    avg_sentence_length = round(word_count / sentence_count, 2) if sentence_count > 0 else 0

    scenario_id = filename.replace(".txt", "")
    rows.append({
        "Scenario ID": scenario_id,
        "Word Count": word_count,
        "Sentence Count": sentence_count,
        "average_sentence_length": avg_sentence_length
    })

# Build Excel with formatting
wb = Workbook()
ws = wb.active
ws.title = "Results"

headers = ["Scenario ID", "Word Count", "Sentence Count", "average_sentence_length"]
header_fill = PatternFill("solid", start_color="2D5016")  # dark green like the image
header_font = Font(bold=True, color="FFFFFF", name="Arial")

for col_idx, header in enumerate(headers, start=1):
    cell = ws.cell(row=1, column=col_idx, value=header)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center")

# Alternating row colors
fill_light_green = PatternFill("solid", start_color="D9EAD3")
fill_light_blue  = PatternFill("solid", start_color="CFE2F3")
fill_white       = PatternFill("solid", start_color="FFFFFF")
row_fills = [fill_light_green, fill_light_blue, fill_white]

for row_idx, row_data in enumerate(rows, start=2):
    fill = row_fills[(row_idx - 2) % len(row_fills)]
    for col_idx, key in enumerate(headers, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=row_data[key])
        cell.fill = fill
        cell.font = Font(name="Arial")
        cell.alignment = Alignment(horizontal="center")

# Column widths
ws.column_dimensions["A"].width = 18
ws.column_dimensions["B"].width = 14
ws.column_dimensions["C"].width = 16
ws.column_dimensions["D"].width = 24

output_file = "results.xlsx"
wb.save(output_file)
print(f"Done! Saved as {output_file}")

# Preview
df = pd.DataFrame(rows)
display(df)

In [ ]:
# Cell 4: Download the Excel file
files.download("results.xlsx")